# Model Monitoring & Automated Re-training

**Session 4 — Deep Dive: Deployment, Inference & MLOps**  
**Notebook 3 of 3** | ~20 minutes

---

## What You Will Learn

| Concept | What We Cover |
|---------|---------------|
| **Auto-Capture Data** | Query the inference table to see logged requests/responses |
| **Model Monitor** | Create a monitor to track drift and performance over time |
| **Drift Metrics** | PSI (Population Stability Index) for distribution shifts |
| **Snowflake Alert** | Trigger automated re-training when drift exceeds threshold |

---

## MLOps Feedback Loop

```
 ┌─────────┐     ┌──────────┐     ┌──────────────┐     ┌───────────┐
 │  Model  │────▶│ Service  │────▶│ Auto-Capture │────▶│  Monitor  │
 │Registry │     │ (SPCS)   │     │ (Inference   │     │  (Drift)  │
 │         │     │          │     │   Table)     │     │           │
 └────▲────┘     └──────────┘     └──────────────┘     └─────┬─────┘
      │                                                       │
      │         ┌──────────────┐     ┌──────────────┐         │
      └─────────│  Re-train    │◀────│    Alert     │◀────────┘
                │  Pipeline    │     │  (Threshold) │
                └──────────────┘     └──────────────┘
```

## 1 | Environment Setup

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import json

from snowflake.ml.registry import Registry
from snowflake.snowpark import functions as F

from utils import get_config
from utils import get_session

config = get_config("config.yaml")
DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
WH = config.snowflake.warehouse
MODEL_NAME = config.model.model_name
SERVICE_NAME = config.deploy.service_name
MONITOR_NAME = config.monitor.monitor_name

session = get_session(
    config.snowflake.connection_name,
    database_name=DB,
    schema_name=SCHEMA,
    warehouse_name=WH,
)
reg = Registry(session=session, database_name=DB, schema_name=SCHEMA)
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")
print(f"Model: {MODEL_NAME} | Monitor: {MONITOR_NAME}")

## 2 | Querying Auto-Captured Inference Data

When a service has `autocapture=True`, every request/response is logged to the model's
**inference table**. This is accessible via the `INFERENCE_TABLE()` function.

### What Gets Captured

| Field | Description |
|-------|-------------|
| `snow.model_serving.request.data.<col>` | Input features sent to the model |
| `snow.model_serving.response.data.<col>` | Inference output returned |
| `snow.model_serving.request.timestamp` | When the request was received |
| `snow.model_serving.response.code` | HTTP status code |
| `snow.model_serving.function.name` | Which function was called |

In [ ]:
session.sql(f"""
    SELECT OBSERVED_TIMESTAMP, RESOURCE_ATTRIBUTES, RECORD_ATTRIBUTES
    FROM TABLE(INFERENCE_TABLE('{MODEL_NAME}'))
    ORDER BY TIMESTAMP DESC
    LIMIT 10
""").show()

### Inferences Per Hour

In [ ]:
session.sql(f"""
    SELECT
        DATE_TRUNC('hour', TIMESTAMP) AS HOUR,
        COUNT(*) AS REQUEST_COUNT
    FROM TABLE(
        INFERENCE_TABLE(
            '{MODEL_NAME}',
            SERVICE => '{SERVICE_NAME}'
        )
    )
    WHERE TIMESTAMP > DATEADD('day', -7, CURRENT_TIMESTAMP())
    GROUP BY 1
    ORDER BY 1
""").show()

### Predictions Per Class

In [ ]:
session.sql(f"""
    SELECT
        RECORD_ATTRIBUTES:"snow.model_serving.response.data.output_feature_0"::STRING AS PREDICTED_CLASS,
        COUNT(*) AS PREDICTION_COUNT
    FROM TABLE(INFERENCE_TABLE('{MODEL_NAME}'))
    WHERE RECORD_ATTRIBUTES:"snow.model_serving.response.code" = 200
      AND RECORD_ATTRIBUTES:"snow.model_serving.function.name" = 'predict'
      AND RESOURCE_ATTRIBUTES:"snow.model.version.name" = 'V_20260528_011030'
    GROUP BY 1
    ORDER BY 1
""").show()

### Inferences Per Model Version

In [ ]:
session.sql(f"""
    SELECT
        RESOURCE_ATTRIBUTES:"snow.model.version.name"::STRING AS MODEL_VERSION,
        COUNT(*) AS INFERENCE_COUNT
    FROM TABLE(INFERENCE_TABLE('{MODEL_NAME}'))
    WHERE RECORD_ATTRIBUTES:"snow.model_serving.response.code" = 200
      AND RECORD_ATTRIBUTES:"snow.model_serving.function.name" = 'predict'
    GROUP BY 1
    ORDER BY 1
""").show()

### Filter by Gateway

If using a gateway, you can filter inference logs by gateway name:

In [ ]:
GATEWAY_NAME = "PATIENT_RISK_GATEWAY"

session.sql(f"""
    SELECT
        TIMESTAMP,
        RESOURCE_ATTRIBUTES:"snow.model.version.name"::STRING AS MODEL_VERSION,
        RECORD_ATTRIBUTES:"snow.model_serving.response.data.output_feature_0"::STRING AS PREDICTION
    FROM TABLE(
        INFERENCE_TABLE(
            '{MODEL_NAME}',
            GATEWAY => '{GATEWAY_NAME}'
        )
    )
    ORDER BY TIMESTAMP DESC
    LIMIT 5
""").show()

## 3 | Prepare Monitor Source Table

The model monitor needs a **source table** with:
- A `TIMESTAMP_NTZ` column
- Prediction columns
- (Optional) Feature columns for drift detection

We'll create a view over the inference table that flattens the captured data:

In [ ]:
INFERENCE_VIEW = config.monitor.inference_logs_view

session.sql(f"""
    CREATE OR REPLACE VIEW {DB}.{SCHEMA}.{INFERENCE_VIEW} AS
    SELECT
        TIMESTAMP::TIMESTAMP_NTZ AS INFERENCE_TIMESTAMP,
        RECORD_ATTRIBUTES:"snow.model_serving.request.data.AGE"::FLOAT AS AGE,
        RECORD_ATTRIBUTES:"snow.model_serving.request.data.BMI"::FLOAT AS BMI,
        RECORD_ATTRIBUTES:"snow.model_serving.request.data.HEART_RATE"::FLOAT AS HEART_RATE,
        RECORD_ATTRIBUTES:"snow.model_serving.request.data.SYSTOLIC_BP"::FLOAT AS SYSTOLIC_BP,
        RECORD_ATTRIBUTES:"snow.model_serving.request.data.DIASTOLIC_BP"::FLOAT AS DIASTOLIC_BP,
        RECORD_ATTRIBUTES:"snow.model_serving.request.data.OXYGEN_SATURATION"::FLOAT AS OXYGEN_SATURATION,
        RECORD_ATTRIBUTES:"snow.model_serving.request.data.VITAL_SIGNS_SEVERITY"::STRING AS VITAL_SIGNS_SEVERITY,
        RECORD_ATTRIBUTES:"snow.model_serving.request.data.ADMISSION_TYPE"::STRING AS ADMISSION_TYPE,
        RECORD_ATTRIBUTES:"snow.model_serving.request.data.BMI_CATEGORY"::STRING AS BMI_CATEGORY,
        RECORD_ATTRIBUTES:"snow.model_serving.response.data.output_feature_0"::STRING AS RISK_LEVEL
    FROM TABLE(INFERENCE_TABLE('{MODEL_NAME}'))
    WHERE RECORD_ATTRIBUTES:"snow.model_serving.response.code" = 200
      AND RECORD_ATTRIBUTES:"snow.model_serving.function.name" = 'predict'
""").collect()

print(f"Created view: {DB}.{SCHEMA}.{INFERENCE_VIEW}")
session.sql(f"SELECT * FROM {DB}.{SCHEMA}.{INFERENCE_VIEW} LIMIT 5").show()

## 4 | Create Baseline Table

The baseline represents the **expected distribution** of your model's inputs/outputs
at deployment time. The monitor compares current data against this baseline to detect drift.

In [ ]:
BASELINE_TABLE = config.monitor.baseline_table

model = reg.get_model(MODEL_NAME)
mv = model.last()

# == Get immutable dataset used to train model ==
upstream_datasets = mv.lineage(direction="upstream", domain_filter=["dataset"])
print(f"Training dataset: {upstream_datasets[0].fully_qualified_name}")

ds = upstream_datasets[0]
baseline_df = ds.read.to_snowpark_dataframe()

baseline_df.select(
    F.current_timestamp().cast("TIMESTAMP_NTZ").alias("INFERENCE_TIMESTAMP"),
    F.col("AGE").cast("FLOAT").alias("AGE"),
    F.col("BMI").cast("FLOAT").alias("BMI"),
    F.col("HEART_RATE").cast("FLOAT").alias("HEART_RATE"),
    F.col("SYSTOLIC_BP").cast("FLOAT").alias("SYSTOLIC_BP"),
    F.col("DIASTOLIC_BP").cast("FLOAT").alias("DIASTOLIC_BP"),
    F.col("OXYGEN_SATURATION").cast("FLOAT").alias("OXYGEN_SATURATION"),
    F.col("VITAL_SIGNS_SEVERITY").cast("STRING").alias("VITAL_SIGNS_SEVERITY"),
    F.col("ADMISSION_TYPE").cast("STRING").alias("ADMISSION_TYPE"),
    F.col("BMI_CATEGORY").cast("STRING").alias("BMI_CATEGORY"),
    F.col("RISK_LEVEL").cast("STRING").alias("RISK_LEVEL"),
).write.save_as_table(f"{DB}.{SCHEMA}.{BASELINE_TABLE}", mode="overwrite")

count = session.sql(f"SELECT COUNT(*) AS N FROM {DB}.{SCHEMA}.{BASELINE_TABLE}").collect()[0]["N"]
print(f"Baseline table created: {DB}.{SCHEMA}.{BASELINE_TABLE} ({count} rows)")

## 5 | Create Model Monitor

The monitor tracks:
- **Drift** — PSI (Population Stability Index) comparing current vs baseline distributions
- **Statistics** — Count, null counts, distributions over time

```sql
CREATE MODEL MONITOR <name> WITH
    MODEL = <model>          -- registered model
    VERSION = '<version>'    -- version to monitor
    SOURCE = <table/view>    -- where predictions land
    BASELINE = <table>       -- reference distribution
    ...
```

In [ ]:
reg = Registry(session=session, database_name=DB, schema_name=SCHEMA)
model = reg.get_model(MODEL_NAME)
mv = model.default

monitor_sql = f"""
    CREATE OR REPLACE MODEL MONITOR {DB}.{SCHEMA}.{MONITOR_NAME} WITH
        MODEL = {DB}.{SCHEMA}.{MODEL_NAME}
        VERSION = '{mv.version_name}'
        FUNCTION = 'PREDICT'
        SOURCE = {DB}.{SCHEMA}.{INFERENCE_VIEW}
        BASELINE = {DB}.{SCHEMA}.{BASELINE_TABLE}
        WAREHOUSE = {WH}
        REFRESH_INTERVAL = '1 hour'
        AGGREGATION_WINDOW = '1 day'
        TIMESTAMP_COLUMN = INFERENCE_TIMESTAMP
        PREDICTION_CLASS_COLUMNS = ('RISK_LEVEL')
        SEGMENT_COLUMNS = ('ADMISSION_TYPE', 'BMI_CATEGORY')
"""

print("Creating model monitor...")
print(monitor_sql)
session.sql(monitor_sql).collect()
print(f"\nMonitor '{MONITOR_NAME}' created successfully!")

In [ ]:
session.sql(f"DESC MODEL MONITOR {DB}.{SCHEMA}.{MONITOR_NAME}").show()

## 6 | Query Drift Metrics

Once the monitor has run at least one refresh cycle, you can query drift metrics:

| Metric | Description |
|--------|-------------|
| **PSI** | Population Stability Index — measures distribution shift |
| **KL_DIVERGENCE** | Kullback-Leibler divergence |

### PSI Interpretation

| PSI Value | Interpretation |
|-----------|----------------|
| < 0.10 | No significant drift |
| 0.10 - 0.25 | Moderate drift — investigate |
| > 0.25 | Significant drift — consider retraining |

In [ ]:
session.sql(f"""
    SELECT *
    FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
        '{MONITOR_NAME}',
        'POPULATION_STABILITY_INDEX',
        'HEART_RATE',
        '1 DAY',
        DATEADD('day', -30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
    ))
    ORDER BY 1 DESC
""").show()

In [ ]:
session.sql(f"""
    SELECT *
    FROM TABLE(MODEL_MONITOR_STAT_METRIC(
        '{MONITOR_NAME}',
        'COUNT',
        'RISK_LEVEL',
        '1 DAY',
        DATEADD('day', -30, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
    ))
    ORDER BY 1 DESC
""").show()

## 7 | Create Drift Alert for Automated Re-training

A Snowflake **Alert** periodically checks a condition and executes an action when triggered.
We'll create an alert that:

1. Checks PSI drift on the `RISK_LEVEL` prediction column
2. If PSI > threshold → triggers the re-training pipeline (resumes the root task)

```
 Alert (cron)  ──check──▶  PSI > 0.25?  ──yes──▶  EXECUTE TASK (retrain)
                                          └─ no ──▶  (do nothing)
```

In [ ]:
drift_config = config.monitor.drift_alerts[0]
ALERT_NAME = drift_config.alert_name
DRIFT_THRESHOLD = drift_config.drift_threshold
DRIFT_COLUMN = drift_config.column
RETRAIN_TASK = config.monitor.retrain_root_task

alert_sql = f"""
CREATE OR REPLACE ALERT {DB}.{SCHEMA}.{ALERT_NAME}
    WAREHOUSE = {WH}
    SCHEDULE = '{drift_config.schedule}'
    IF (EXISTS (
        SELECT MAX(metric_value) AS MAX_DRIFT
        FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
            '{MONITOR_NAME}',
            'POPULATION_STABILITY_INDEX',
            '{DRIFT_COLUMN}',
            '1 DAY',
            DATEADD('day', -1, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
            CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
        ))
        WHERE metric_value > {DRIFT_THRESHOLD}
    ))
    THEN
        EXECUTE TASK {DB}.{SCHEMA}.{RETRAIN_TASK}
"""

print("Creating drift alert...")
print(alert_sql)
session.sql(alert_sql).collect()
print(f"\nAlert '{ALERT_NAME}' created!")
print(f"  Schedule: {drift_config.schedule}")
print(f"  Threshold: PSI > {DRIFT_THRESHOLD} on column '{DRIFT_COLUMN}'")
print(f"  Action: Execute task '{RETRAIN_TASK}'")

In [ ]:
session.sql(f"ALTER ALERT {DB}.{SCHEMA}.{ALERT_NAME} RESUME").collect()
print(f"Alert '{ALERT_NAME}' is now ACTIVE.")

### Additional Alert: Feature Drift

We can also monitor drift on input features (not just predictions):

In [ ]:
if len(config.monitor.drift_alerts) > 1:
    drift_config2 = config.monitor.drift_alerts[1]
    ALERT_NAME_2 = drift_config2.alert_name

    alert2_sql = f"""
    CREATE OR REPLACE ALERT {DB}.{SCHEMA}.{ALERT_NAME_2}
        WAREHOUSE = {WH}
        SCHEDULE = '{drift_config2.schedule}'
        IF (EXISTS (
            SELECT MAX(metric_value) AS MAX_DRIFT
            FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
                '{MONITOR_NAME}',
                'POPULATION_STABILITY_INDEX',
                '{drift_config2.column}',
                '1 DAY',
                DATEADD('day', -1, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
                CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
            ))
            WHERE metric_value > {drift_config2.drift_threshold}
        ))
        THEN
            EXECUTE TASK {DB}.{SCHEMA}.{RETRAIN_TASK}
    """

    session.sql(alert2_sql).collect()
    session.sql(f"ALTER ALERT {DB}.{SCHEMA}.{ALERT_NAME_2} RESUME").collect()
    print(f"Alert '{ALERT_NAME_2}' created and activated.")
    print(f"  Monitors: PSI > {drift_config2.drift_threshold} on '{drift_config2.column}'")
else:
    print("No additional drift alerts configured.")

## 8 | Verify Alerts

In [ ]:
session.sql(f"SHOW ALERTS IN SCHEMA {DB}.{SCHEMA}").show()

## 9 | Management Commands Reference

### Monitor Management

| Operation | Command |
|-----------|--------|
| Suspend | `ALTER MODEL MONITOR <name> SUSPEND;` |
| Resume | `ALTER MODEL MONITOR <name> RESUME;` |
| Set baseline | `ALTER MODEL MONITOR <name> SET BASELINE = '<table>';` |
| Drop | `DROP MODEL MONITOR <name>;` |

### Alert Management

| Operation | Command |
|-----------|--------|
| Suspend | `ALTER ALERT <name> SUSPEND;` |
| Resume | `ALTER ALERT <name> RESUME;` |
| Check history | `SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.ALERT_HISTORY;` |
| Drop | `DROP ALERT <name>;` |

In [ ]:
# session.sql(f"ALTER MODEL MONITOR {DB}.{SCHEMA}.{MONITOR_NAME} SUSPEND").collect()
# session.sql(f"ALTER MODEL MONITOR {DB}.{SCHEMA}.{MONITOR_NAME} RESUME").collect()
# session.sql(f"DROP MODEL MONITOR {DB}.{SCHEMA}.{MONITOR_NAME}").collect()
# session.sql(f"ALTER ALERT {DB}.{SCHEMA}.{ALERT_NAME} SUSPEND").collect()
# session.sql(f"DROP ALERT {DB}.{SCHEMA}.{ALERT_NAME}").collect()

## Summary

In this notebook we completed the MLOps feedback loop:

1. **Queried Auto-Capture data** — Inspected logged inference requests/responses from `INFERENCE_TABLE()`
2. **Created a Monitor source view** — Flattened inference data into a structured format
3. **Built a baseline table** — Captured the reference distribution at deployment time
4. **Created a Model Monitor** — Tracks PSI drift on predictions daily
5. **Set up Drift Alerts** — Automatically triggers re-training when PSI exceeds threshold

---

## Session 4 Complete!

Across the three notebooks in this session, we covered:

| Notebook | Topic |
|----------|-------|
| **01** | Model Deployment — SPCS service + Gateway with auto-capture |
| **02** | Inference — SQL (warehouse) + REST API (real-time) |
| **03** | Monitoring — Drift detection + automated re-training alerts |

Together with Sessions 1-3, this completes the **end-to-end ML lifecycle** on Snowflake:

```
 Data → Features → Training → Registry → Deployment → Inference → Monitoring → Re-training
  S1       S2         S3         S2          S4           S4          S4           S4
```